# 🧠 3. Protein Embeddings Generation

Este notebook es el núcleo computacional del pipeline de **ProtFlash**. En esta etapa, las secuencias de aminoácidos se transforman en vectores numéricos densos (embeddings) que capturan las propiedades biológicas y estructurales de las proteínas.

---

## 📋 Descripción General
El script utiliza modelos de lenguaje pre-entrenados para convertir las secuencias de texto en representaciones de **768 dimensiones**. Este proceso es fundamental para que los modelos de Deep Learning puedan "entender" la relación entre diferentes aminoácidos en una secuencia.



## 🛠️ Especificaciones Técnicas

1.  **Modelo y Arquitectura:**
    * Basado en modelos de lenguaje específicos para proteínas (ProtBert/ProtFlash).
    * Utiliza un método de **Mean Pooling** sobre los tokens para obtener una representación única por cada proteína.
2.  **Configuración de Hardware:**
    * Optimizado para ejecutarse en **GPU (T4)** mediante el uso de tensores y aceleración CUDA.
3.  **Procesamiento Eficiente:**
    * Implementa **Batch Processing** (tamaño de lote de 64) para maximizar el uso de la memoria de video y acelerar la inferencia.
4.  **Flujo de Trabajo:**
    * Carga los archivos `train.csv`, `val.csv` y `test.csv` generados anteriormente.
    * Genera embeddings para un total de **40,358 secuencias**.
    * Realiza el seguimiento del progreso en tiempo real mediante barras de porcentaje.

## 💾 Salida de Datos
Los resultados se exportan como arreglos de NumPy (`.npy`) y archivos de etiquetas, listos para ser consumidos por el clasificador final:
* `train_embeddings.npy` / `train_labels.npy`
* `val_embeddings.npy` / `val_labels.npy`
* `test_embeddings.npy` / `test_labels.npy`

Configuracion de rutas

In [ ]:
INPUT_CSV = "/content/drive/MyDrive/Experimentos/ProtFlash/Data/csv"
OUTPUT_EMBEDDINGS = "/content/drive/MyDrive/Experimentos/ProtFlash/Embeddings"

Instalacion de librerias

In [ ]:
pip install ProtFlash

Importacion de librerias

In [ ]:
from ProtFlash.pretrain import load_prot_flash_base
from ProtFlash.utils import batchConverter
import torch
import gc
import numpy as np
import pandas as pd
import os

Carga de datos

In [ ]:
df_train = pd.read_csv(os.path.join(INPUT_CSV, "train.csv"))
df_val = pd.read_csv(os.path.join(INPUT_CSV, "val.csv"))
df_test = pd.read_csv(os.path.join(INPUT_CSV, "test.csv"))

train = []
val = []
test = []

In [ ]:
df_train.columns

Index(['Unnamed: 0', 'id', 'sequence', 'label'], dtype='object')

In [ ]:
#Format datasets
for idx, row in df_train.iterrows():
  train.append((row["id"],row['sequence']))

for idx, row in df_val.iterrows():
  val.append((row["id"],row['sequence']))

for idx, row in df_test.iterrows():
  test.append((row["id"],row['sequence']))

Carga del modelo ProtFlash

In [ ]:
model = load_prot_flash_base()
total = len(train)
batch_size = 64
device = 'cuda'
all_embeddings = []
all_ids = []
model.to(device)
model.eval()
print("-----------")

-----------


**Train**

In [ ]:
for i in range(0, total, batch_size):
  batch_data = train[i:i + batch_size]
  ids, batch_token, lengths = batchConverter(batch_data)

  # Mover tensores a GPU
  batch_token = batch_token.to(device)
  lengths = lengths.to(device)

  with torch.no_grad():
      token_embedding = model(batch_token, lengths)
  # Generate per-sequence representations via averaging
  #print("POOL")
  embeddings_cpu = token_embedding.cpu().numpy()
  for j, (_, seq) in enumerate(batch_data):
    all_embeddings.append(embeddings_cpu[j, 0: len(seq) + 1].mean(0))

  all_ids.extend(ids)

  del batch_token, token_embedding, ids, lengths, batch_data
  torch.cuda.empty_cache()
  gc.collect()

  # Progreso correcto
  procesado = min(i + batch_size, total)
  print(f"   Procesado: {procesado}/{total} ({100 * procesado / total:.1f}%)")

   Procesado: 64/19548 (0.3%)
   Procesado: 128/19548 (0.7%)
   Procesado: 192/19548 (1.0%)
   Procesado: 256/19548 (1.3%)
   Procesado: 320/19548 (1.6%)
   Procesado: 384/19548 (2.0%)
   Procesado: 448/19548 (2.3%)
   Procesado: 512/19548 (2.6%)
   Procesado: 576/19548 (2.9%)
   Procesado: 640/19548 (3.3%)
   Procesado: 704/19548 (3.6%)
   Procesado: 768/19548 (3.9%)
   Procesado: 832/19548 (4.3%)
   Procesado: 896/19548 (4.6%)
   Procesado: 960/19548 (4.9%)
   Procesado: 1024/19548 (5.2%)
   Procesado: 1088/19548 (5.6%)
   Procesado: 1152/19548 (5.9%)
   Procesado: 1216/19548 (6.2%)
   Procesado: 1280/19548 (6.5%)
   Procesado: 1344/19548 (6.9%)
   Procesado: 1408/19548 (7.2%)
   Procesado: 1472/19548 (7.5%)
   Procesado: 1536/19548 (7.9%)
   Procesado: 1600/19548 (8.2%)
   Procesado: 1664/19548 (8.5%)
   Procesado: 1728/19548 (8.8%)
   Procesado: 1792/19548 (9.2%)
   Procesado: 1856/19548 (9.5%)
   Procesado: 1920/19548 (9.8%)
   Procesado: 1984/19548 (10.1%)
   Procesado: 2048/1954

Guardar

In [ ]:
# Convertir embeddings a array numpy
all_embeddings = np.array(all_embeddings)

df_embeddings = pd.DataFrame({
    "seq_id": all_ids,
    "embedding": [emb.tolist() for emb in all_embeddings]
})

# Guardar
output_path = f"{OUTPUT_EMBEDDINGS}/train_embeddings.csv"
df_embeddings.to_csv(output_path, index=False)
print(f"✅ Guardado: {output_path} → {df_embeddings.shape}")

✅ Guardado: /content/drive/MyDrive/ProtFlash-2102/Embeddings/train_embeddings.csv → (19548, 2)


**Validation**

In [ ]:
total = len(val)
all_embeddings = []
all_ids = []

In [ ]:
for i in range(0, total, batch_size):
  batch_data = val[i:i + batch_size]
  ids, batch_token, lengths = batchConverter(batch_data)

  # Mover tensores a GPU
  batch_token = batch_token.to(device)
  lengths = lengths.to(device)

  with torch.no_grad():
      token_embedding = model(batch_token, lengths)
  # Generate per-sequence representations via averaging
  embeddings_cpu = token_embedding.cpu().numpy()
  for j, (_, seq) in enumerate(batch_data):
    all_embeddings.append(embeddings_cpu[j, 0: len(seq) + 1].mean(0))

  all_ids.extend(ids)

  del batch_token, token_embedding, ids, lengths, batch_data
  torch.cuda.empty_cache()
  gc.collect()

  # Progreso correcto
  procesado = min(i + batch_size, total)
  print(f"   Procesado: {procesado}/{total} ({100 * procesado / total:.1f}%)")

   Procesado: 64/5125 (1.2%)
   Procesado: 128/5125 (2.5%)
   Procesado: 192/5125 (3.7%)
   Procesado: 256/5125 (5.0%)
   Procesado: 320/5125 (6.2%)
   Procesado: 384/5125 (7.5%)
   Procesado: 448/5125 (8.7%)
   Procesado: 512/5125 (10.0%)
   Procesado: 576/5125 (11.2%)
   Procesado: 640/5125 (12.5%)
   Procesado: 704/5125 (13.7%)
   Procesado: 768/5125 (15.0%)
   Procesado: 832/5125 (16.2%)
   Procesado: 896/5125 (17.5%)
   Procesado: 960/5125 (18.7%)
   Procesado: 1024/5125 (20.0%)
   Procesado: 1088/5125 (21.2%)
   Procesado: 1152/5125 (22.5%)
   Procesado: 1216/5125 (23.7%)
   Procesado: 1280/5125 (25.0%)
   Procesado: 1344/5125 (26.2%)
   Procesado: 1408/5125 (27.5%)
   Procesado: 1472/5125 (28.7%)
   Procesado: 1536/5125 (30.0%)
   Procesado: 1600/5125 (31.2%)
   Procesado: 1664/5125 (32.5%)
   Procesado: 1728/5125 (33.7%)
   Procesado: 1792/5125 (35.0%)
   Procesado: 1856/5125 (36.2%)
   Procesado: 1920/5125 (37.5%)
   Procesado: 1984/5125 (38.7%)
   Procesado: 2048/5125 (40.0%)

Guardar

In [ ]:
# Convertir embeddings a array numpy
all_embeddings = np.array(all_embeddings)

df_embeddings = pd.DataFrame({
    "seq_id": all_ids,
    "embedding": [emb.tolist() for emb in all_embeddings]
})

# Guardar
output_path = f"{OUTPUT_EMBEDDINGS}/val_embeddings.csv"
df_embeddings.to_csv(output_path, index=False)
print(f"✅ Guardado: {output_path} → {df_embeddings.shape}")

**Test**

In [ ]:
total = len(test)
all_embeddings = []
all_ids = []

In [ ]:
for i in range(0, total, batch_size):
  batch_data = test[i:i + batch_size]
  ids, batch_token, lengths = batchConverter(batch_data)

  # Mover tensores a GPU
  batch_token = batch_token.to(device)
  lengths = lengths.to(device)

  with torch.no_grad():
      token_embedding = model(batch_token, lengths)
  # Generate per-sequence representations via averaging
  embeddings_cpu = token_embedding.cpu().numpy()
  for j, (_, seq) in enumerate(batch_data):
    all_embeddings.append(embeddings_cpu[j, 0: len(seq) + 1].mean(0))

  all_ids.extend(ids)

  del batch_token, token_embedding, ids, lengths, batch_data
  torch.cuda.empty_cache()
  gc.collect()

  # Progreso correcto
  procesado = min(i + batch_size, total)
  print(f"   Procesado: {procesado}/{total} ({100 * procesado / total:.1f}%)")

   Procesado: 64/15685 (0.4%)
   Procesado: 128/15685 (0.8%)
   Procesado: 192/15685 (1.2%)
   Procesado: 256/15685 (1.6%)
   Procesado: 320/15685 (2.0%)
   Procesado: 384/15685 (2.4%)
   Procesado: 448/15685 (2.9%)
   Procesado: 512/15685 (3.3%)
   Procesado: 576/15685 (3.7%)
   Procesado: 640/15685 (4.1%)
   Procesado: 704/15685 (4.5%)
   Procesado: 768/15685 (4.9%)
   Procesado: 832/15685 (5.3%)
   Procesado: 896/15685 (5.7%)
   Procesado: 960/15685 (6.1%)
   Procesado: 1024/15685 (6.5%)
   Procesado: 1088/15685 (6.9%)
   Procesado: 1152/15685 (7.3%)
   Procesado: 1216/15685 (7.8%)
   Procesado: 1280/15685 (8.2%)
   Procesado: 1344/15685 (8.6%)
   Procesado: 1408/15685 (9.0%)
   Procesado: 1472/15685 (9.4%)
   Procesado: 1536/15685 (9.8%)
   Procesado: 1600/15685 (10.2%)
   Procesado: 1664/15685 (10.6%)
   Procesado: 1728/15685 (11.0%)
   Procesado: 1792/15685 (11.4%)
   Procesado: 1856/15685 (11.8%)
   Procesado: 1920/15685 (12.2%)
   Procesado: 1984/15685 (12.6%)
   Procesado: 204

Guardar

In [ ]:
# Convertir embeddings a array numpy
all_embeddings = np.array(all_embeddings)

df_embeddings = pd.DataFrame({
    "seq_id": all_ids,
    "embedding": [emb.tolist() for emb in all_embeddings]
})

# Guardar
output_path = f"{OUTPUT_EMBEDDINGS}/test_embeddings.csv"
df_embeddings.to_csv(output_path, index=False)
print(f"✅ Guardado: {output_path} → {df_embeddings.shape}")

✅ Guardado: /content/drive/MyDrive/ProtFlash-2102/Embeddings/test_embeddings.csv → (15685, 2)
